# פרק 14 — ניהול פרויקט במדעי הנתונים | מעבדת תלמיד
**מהדורת תלמיד • מסלול ההתמחות בלמידה עמוקה (25 שעות בתכנון)**

המטרה: להפוך ניסוי סיווג למיני־מערכת שניתנת להרצה חוזרת ולבדיקה. הדוגמה משתמשת בנתונים ציבוריים מתוך `scikit-learn`, ואינה דורשת הורדה או פרטים אישיים. **משימות התלמיד אינן פתורות.**

סדר עבודה: רצוי להריץ את התאים המודרכים בסדרם. אל תשתמשו ב־Test כדי לבחור מודל. הקוד שבפרק הוא מודל הדגמה — בפרויקט גמר תידרש עבודה עצמאית מורכבת יותר.

## תא מודרך 1 — הכנה, מקור הנתונים, וצורות המערכים

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from pathlib import Path
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

digits = load_digits()
X, y = digits.data, digits.target
print('Rows, features:', X.shape)
print('Classes:', np.unique(y))
print('Python package version (scikit-learn):', sklearn.__version__)
assert X.shape[1] == 64

## תא מודרך 2 — חלוקה למבחן חיצוני שאינו משתתף בחיפוש

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Train:', len(y_train), 'Test:', len(y_test))

## תא מודרך 3 — Baseline ומודל ראשון (להמחשה)

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))
model.fit(X_train, y_train)
print('Baseline train accuracy:', accuracy_score(y_train, baseline.predict(X_train)))
print('Model train accuracy:', accuracy_score(y_train, model.predict(X_train)))
# אין להסיק מציוני אימון על הכללה לנתונים חדשים.

## תא מודרך 4 — הערכת CV פנימית על Train בלבד

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    X_train, y_train, cv=cv, scoring='f1_macro'
)
print('Fold scores:', np.round(cv_scores, 4))
print('Mean CV Macro-F1:', np.mean(cv_scores))

## תא מודרך 5 — חיזוי עם בדיקות קלט

In [ ]:
def predict_digit(values, trained_model):
    arr = np.asarray(values, dtype=float)
    if arr.shape != (64,):
        raise ValueError('Expected a vector of 64 elements')
    if not np.isfinite(arr).all():
        raise ValueError('Missing or infinite value')
    if np.any(arr < 0) or np.any(arr > 16):
        raise ValueError('Pixel values out of range 0..16')
    return int(trained_model.predict(arr.reshape(1, -1))[0])

print('Predicted digit:', predict_digit(X_train[0], model))

## תא מודרך 6 — שמירת תיעוד ושחזור תחזית (קובץ שנוצר כאן בלבד)

In [ ]:
import json
import joblib

out = Path('chapter14_artifacts')
out.mkdir(exist_ok=True)
artifact = out / 'digit_model.joblib'
joblib.dump(model, artifact)
# אזהרה: טוענים רק קובץ שיצרנו בעצמנו או קובץ שמקורו מהימן ומאומת.
restored = joblib.load(artifact)
assert predict_digit(X_train[0], model) == predict_digit(X_train[0], restored)
info = {
    'dataset': 'sklearn digits',
    'features': 64,
    'random_state': 42,
    'test_size': 0.20,
    'sklearn_version': sklearn.__version__
}
(out / 'model_info.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
print('Saved local artifacts:', [p.name for p in out.iterdir()])

---
## משימות תלמיד עצמאיות — ללא פתרונות
בכל משימה יש לכתוב גם **מה עשיתי, מדוע, מה הייתה התוצאה ומה המגבלה שלה**. תאי הקוד יישארו חלקיים כדי שתשלימו בעצמכם.

### משימה 1 — כרטיס פרויקט
כתבו ב־Markdown מי המשתמש, מה הקלט והפלט, איזו החלטה מתקבלת ומהי מגבלת השימוש.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# כתבו את כרטיס הפרויקט בתא Markdown חדש.

### משימה 2 — בדיקת מאגר
בדקו כמה תמונות יש מכל ספרה והציגו תרשים שכיחויות. הסבירו מה משמעות חוסר איזון אפשרי.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: counts = ...
# TODO: plot ...

### משימה 3 — מחקר על קבוצות הנתונים
חשבו את מספר הדוגמאות בכל קבוצה והסבירו למה אי אפשר לבחור k לפי Test.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: sizes and explanation ...

### משימה 4 — השוואת Baseline
חשבו Macro F1 של מודל בסיס על קיפולי CV של Train והשוו למסווג אחר.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: cross_val_score for DummyClassifier on X_train ...

### משימה 5 — יומן ניסויים
השוו לפחות שלושה ערכי k ב־KNN באמצעות אותו פרוטוקול CV. תעדו ממוצע וסטיית תקן.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: loop over k values, collect cv scores into a DataFrame ...

### משימה 6 — מניעת זליגת מידע
הסבירו בשתי פסקאות מדוע StandardScaler חייב להיות בתוך Pipeline. הדגימו בקוד תקין.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: your own pipeline ...

### משימה 7 — מבחן סופי
אחרי בחירת k לפי CV, אמנו מחדש על Train והציגו Accuracy, Macro F1 ומטריצת בלבול ב־Test.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: evaluate only after selection is complete ...

### משימה 8 — ניתוח טעויות
מצאו שלוש דוגמאות שנחזו בצורה שגויה, הציגו אותן והסבירו השערות אפשריות לשגיאה.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: find error indices and visualize images ...

### משימה 9 — בדיקות קלט
בדקו כיצד predict_digit מגיבה למערך קצר, ערך שלילי ו־NaN. תעדו תוצאות.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: create invalid examples and test their expected exceptions ...

### משימה 10 — שחזור ניסוי
שמרו גרסת מודל נוספת וקובץ JSON תיעודי. מה צריך לשמור כדי שחבר יוכל לשחזר?

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: metadata keys and reproducibility notes ...

### משימה 11 — ניטור וסיכונים
הציעו שתי בדיקות ניטור שאינן דורשות תוויות אמת, ובדיקה אחת שכן דורשת אותן.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: write textual conclusions in a Markdown cell ...

### משימה 12 — הכנה להגנה בעל־פה
כתבו תמצית 250–350 מילים על ההחלטות שלכם: נתונים, CV, Baseline, מדדים, שגיאות ומגבלה.

**תוצר חובה:** תוצאה או גרף + הסבר במילים שלכם. אין להעתיק פתרון מוכן.

In [ ]:
# TODO: draft your own project defense in Markdown ...

## כרטיס יציאה
1. מדוע מודל שקיבל 100% על Train אינו בהכרח מודל טוב?
2. מה חייב להישמר יחד עם מודל שהעברנו למחשב אחר?
3. איזו תכונה של Test הופכת אותו לבדיקה הוגנת?
4. מה בין Data Drift לבין ירידה במדד F1?

**תזכורת:** אין צורך לאסוף תמונות אישיות או להעלות נתונים רגישים לצורך משימות המחברת.